In [28]:
import pandas as pd 
import numpy as np
import re

In [20]:
df = pd.read_csv('../data/raw/training_and_development_data.csv')

In [21]:
df.head()

,Employee ID,Training Date,Training Program Name,Training Type,Training Outcome,Location,Trainer,Training Duration(Days),Training Cost
0,1001,21-Sep-22,Customer Service,Internal,Failed,Port Greg,Amanda Daniels,4,510.83
1,1002,19-Jul-23,Leadership Development,Internal,Failed,Brandonview,Brittany Chambers,2,582.37
2,1003,24-Feb-23,Technical Skills,Internal,Incomplete,Port Briannahaven,Mark Roberson,4,777.06
3,1004,12-Jan-23,Customer Service,Internal,Completed,Knightborough,Richard Fisher,2,824.30
4,1005,12-May-23,Communication Skills,External,Passed,Bruceshire,Heather Shaffer,4,145.99


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Employee ID              3000 non-null   int64  
 1   Training Date            3000 non-null   object 
 2   Training Program Name    3000 non-null   object 
 3   Training Type            3000 non-null   object 
 4   Training Outcome         3000 non-null   object 
 5   Location                 3000 non-null   object 
 6   Trainer                  3000 non-null   object 
 7   Training Duration(Days)  3000 non-null   int64  
 8   Training Cost            3000 non-null   float64
dtypes: float64(1), int64(2), object(6)
memory usage: 211.1+ KB


In [23]:
df.shape

(3000, 9)

In [24]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Employee ID,3000.0,NaN,NaN,NaN,2500.5,866.169729,1001.0,1750.75,2500.5,3250.25,4000.0
Training Date,3000,366,11-May-23,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Training Program Name,3000,5,Communication Skills,673,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Training Type,3000,2,Internal,1509,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Training Outcome,3000,4,Incomplete,775,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,3000,2738,Smithchester,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Trainer,3000,2942,Michael Smith,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Training Duration(Days),3000.0,NaN,NaN,NaN,2.975667,1.41789,1.0,2.0,3.0,4.0,5.0
Training Cost,3000.0,NaN,NaN,NaN,558.628697,263.217698,100.04,327.5875,572.125,786.9875,999.97


In [25]:
print(df.isnull().sum())

Employee ID                0
Training Date              0
Training Program Name      0
Training Type              0
Training Outcome           0
Location                   0
Trainer                    0
Training Duration(Days)    0
Training Cost              0
dtype: int64


In [26]:
df.duplicated().sum()

np.int64(0)

In [29]:
df.columns = [
    re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-z0-9]+", "_", col.strip().lower())
    ).strip("_")
    for col in df.columns
]

print(df.columns.tolist())

['employee_id', 'training_date', 'training_program_name', 'training_type', 'training_outcome', 'location', 'trainer', 'training_duration_days', 'training_cost']


In [31]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Convert empty strings into missing values
df[text_columns] = df[text_columns].replace("", pd.NA)

In [32]:
df["employee_id"] = pd.to_numeric(
    df["employee_id"],
    errors="coerce"
)

In [33]:
df["training_date"] = pd.to_datetime(
    df["training_date"],
    format="%d-%b-%y",
    errors="coerce"
)

In [34]:
df["training_duration_days"] = pd.to_numeric(
    df["training_duration_days"],
    errors="coerce"
)

In [35]:
df["training_cost"] = pd.to_numeric(
    df["training_cost"],
    errors="coerce"
)

In [36]:
integer_columns = [
    "employee_id",
    "training_duration_days"
]

for col in integer_columns:

    fractional_values = (
        df[col].notna() &
        (df[col] % 1 != 0)
    )

    if fractional_values.any():
        raise ValueError(
            f"Fractional values found in {col}: "
            f"{df.loc[fractional_values, col].head().tolist()}"
        )

    df[col] = df[col].astype("Int64")

In [37]:
# Negative training duration
print(
    "Negative durations:",
    (df["training_duration_days"] < 0).sum()
)

# Negative training cost
print(
    "Negative costs:",
    (df["training_cost"] < 0).sum()
)

# Missing employee IDs
print(
    "Missing employee IDs:",
    df["employee_id"].isna().sum()
)

# Missing training dates
print(
    "Missing training dates:",
    df["training_date"].isna().sum()
)

# Missing categorical values
categorical_columns = [
    "training_program_name",
    "training_type",
    "training_outcome",
    "location",
    "trainer"
]

print("\nCategorical missing values:")
print(df[categorical_columns].isna().sum())

Negative durations: 0
Negative costs: 0
Missing employee IDs: 0
Missing training dates: 0

Categorical missing values:
training_program_name    0
training_type            0
training_outcome         0
location                 0
trainer                  0
dtype: int64


In [38]:
print("Training Date datatype:", df["training_date"].dtype)

print(
    "Total missing dates:",
    df["training_date"].isna().sum()
)

print(
    "Date range:",
    df["training_date"].min(),
    "to",
    df["training_date"].max()
)

Training Date datatype: datetime64[ns]
Total missing dates: 0
Date range: 2022-08-05 00:00:00 to 2023-08-05 00:00:00


In [39]:
duplicate_count = df.duplicated().sum()

print("Exact duplicate records:", duplicate_count)

Exact duplicate records: 0


In [40]:
print("=" * 50)
print("FINAL DATA QUALITY REPORT")
print("=" * 50)

print("\nDataset Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:", df.duplicated().sum())

print("\nStatistical Summary:")
display(df.describe(include="all").T)

print("\nFinal Dataset Preview:")
display(df.head())

FINAL DATA QUALITY REPORT

Dataset Shape: (3000, 9)

Data Types:
employee_id                        Int64
training_date             datetime64[ns]
training_program_name     string[python]
training_type             string[python]
training_outcome          string[python]
location                  string[python]
trainer                   string[python]
training_duration_days             Int64
training_cost                    float64
dtype: object

Missing Values:
employee_id               0
training_date             0
training_program_name     0
training_type             0
training_outcome          0
location                  0
trainer                   0
training_duration_days    0
training_cost             0
dtype: int64

Duplicate Records: 0

Statistical Summary:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
employee_id,3000.0,<NA>,<NA>,<NA>,2500.5,1001.0,1750.75,2500.5,3250.25,4000.0,866.169729
training_date,3000,NaN,NaN,NaN,2023-02-03 08:18:43.200000256,2022-08-05 00:00:00,2022-11-05 00:00:00,2023-02-04 00:00:00,2023-05-03 00:00:00,2023-08-05 00:00:00,NaN
training_program_name,3000,5,Communication Skills,673,NaN,NaN,NaN,NaN,NaN,NaN,NaN
training_type,3000,2,Internal,1509,NaN,NaN,NaN,NaN,NaN,NaN,NaN
training_outcome,3000,4,Incomplete,775,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,3000,2738,Smithchester,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trainer,3000,2942,Michael Smith,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
training_duration_days,3000.0,<NA>,<NA>,<NA>,2.975667,1.0,2.0,3.0,4.0,5.0,1.41789
training_cost,3000.0,NaN,NaN,NaN,558.628697,100.04,327.5875,572.125,786.9875,999.97,263.217698



Final Dataset Preview:


,employee_id,training_date,training_program_name,training_type,training_outcome,location,trainer,training_duration_days,training_cost
0,1001,2022-09-21,Customer Service,Internal,Failed,Port Greg,Amanda Daniels,4,510.83
1,1002,2023-07-19,Leadership Development,Internal,Failed,Brandonview,Brittany Chambers,2,582.37
2,1003,2023-02-24,Technical Skills,Internal,Incomplete,Port Briannahaven,Mark Roberson,4,777.06
3,1004,2023-01-12,Customer Service,Internal,Completed,Knightborough,Richard Fisher,2,824.30
4,1005,2023-05-12,Communication Skills,External,Passed,Bruceshire,Heather Shaffer,4,145.99


In [43]:
df.to_csv('train_an_dev_cleaned.csv', index=False)